# LangGraph Demo: Niche Research Agent

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/knivok/ai-playbook/blob/main/notebooks/01_frameworks/langgraph/05_demo_niche_researcher.ipynb)

---

This is the **benchmark demo** used across all framework chapters. The same task — research a product niche, validate it, produce a report — implemented in LangGraph.

The agent graph looks like this:

```
START
  ↓
research_node  (searches for market data)
  ↓
validate_node  (scores the niche)
  ↓
decision_node  ──→ [score < 40] ──→ reject_node → END
  ↓                                  
[score ≥ 40]
  ↓
report_node  (generates the final report)
  ↓
END
```

Notice the **conditional edge** from `decision_node` — this is what makes LangGraph the right tool here.
A simple chain or CrewAI pipeline cannot express this routing as cleanly.

In [ ]:
!pip install -q langgraph langchain-anthropic langchain-core python-dotenv rich

In [ ]:
import os
from getpass import getpass
from typing import TypedDict, Literal
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, END, START
from rich.console import Console
from rich.panel import Panel
from rich.table import Table

if not os.getenv("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API key: ")

console = Console()
llm = ChatAnthropic(model="claude-3-5-sonnet-20241022", temperature=0)

print("✅ Setup complete")

In [ ]:
# ── 1. Define the State ───────────────────────────────────────────────────────
# State is the shared dictionary that flows through every node.
# Every node can read from it and write new values back.

class NicheResearchState(TypedDict):
    niche: str                    # Input: the niche to research
    research_data: str            # Output of research_node
    viability_score: float        # Output of validate_node
    score_reasoning: str          # Why the score is what it is
    final_report: str             # Output of report_node
    rejection_reason: str         # Output of reject_node (if score too low)
    decision: str                 # "pursue" or "reject"

print("✅ State defined")

In [ ]:
# ── 2. Define Nodes ───────────────────────────────────────────────────────────
# Each node is a plain Python function: takes state, returns partial state update.

def research_node(state: NicheResearchState) -> dict:
    """Research the niche using the LLM as a reasoning engine."""
    console.print(f"\n[cyan]🔍 RESEARCH NODE[/] — Researching: {state['niche']}")

    response = llm.invoke([
        SystemMessage(content="You are a market research analyst. Provide concise, factual market data."),
        HumanMessage(content=f"""
Research the following product niche and provide:
1. Estimated market size (USD billions)
2. Key competitors (top 3)
3. Annual growth rate (%)
4. Competition level (low/medium/high)
5. Key trends driving growth

Niche: {state['niche']}

Be specific. Use real market data where possible.
""")
    ])

    console.print(f"[dim]{response.content[:200]}...[/dim]")
    return {"research_data": response.content}


def validate_node(state: NicheResearchState) -> dict:
    """Score the niche viability based on research data."""
    console.print("\n[yellow]📊 VALIDATE NODE[/] — Scoring viability")

    response = llm.invoke([
        SystemMessage(content="You are a niche validation expert. Output ONLY a JSON object, no other text."),
        HumanMessage(content=f"""
Based on this research data, provide a viability score:

{state['research_data']}

Return ONLY this JSON:
{{
  "viability_score": <number 0-100>,
  "reasoning": "<one sentence explanation>",
  "decision": "pursue" or "reject"
}}

Score > 50 = pursue. Score ≤ 50 = reject.
""")
    ])

    import json, re
    raw = response.content.strip()
    # Strip markdown fences if present
    raw = re.sub(r"```json|```", "", raw).strip()
    parsed = json.loads(raw)

    console.print(f"  Score: [bold]{parsed['viability_score']}[/] — {parsed['reasoning']}")
    return {
        "viability_score": parsed["viability_score"],
        "score_reasoning": parsed["reasoning"],
        "decision": parsed["decision"]
    }


def report_node(state: NicheResearchState) -> dict:
    """Generate the full niche report for viable niches."""
    console.print("\n[green]📝 REPORT NODE[/] — Generating report")

    response = llm.invoke([
        SystemMessage(content="You are a strategic analyst at Knovik. Write clear, actionable reports."),
        HumanMessage(content=f"""
Write a niche validation report for Knovik's product team.

Niche: {state['niche']}
Viability Score: {state['viability_score']}/100
Research Data: {state['research_data']}

Include:
## Executive Summary
## Market Opportunity
## Competitive Landscape
## Recommended Entry Strategy
## Key Risks
## Next Steps for Knovik
""")
    ])

    return {"final_report": response.content}


def reject_node(state: NicheResearchState) -> dict:
    """Handle rejected niches gracefully."""
    console.print("\n[red]❌ REJECT NODE[/] — Niche does not meet threshold")
    reason = f"Viability score {state['viability_score']}/100 is below the 50-point threshold. Reason: {state['score_reasoning']}"
    return {"rejection_reason": reason, "final_report": f"REJECTED: {reason}"}


print("✅ Nodes defined")

In [ ]:
# ── 3. Define the Conditional Edge Router ─────────────────────────────────────
# This is the key LangGraph feature — routing based on runtime state.

def route_after_validation(state: NicheResearchState) -> Literal["report_node", "reject_node"]:
    """Conditional edge: route based on viability score."""
    if state.get("decision") == "pursue" or state.get("viability_score", 0) > 50:
        return "report_node"
    return "reject_node"

print("✅ Router defined")

In [ ]:
# ── 4. Build and Compile the Graph ────────────────────────────────────────────

workflow = StateGraph(NicheResearchState)

# Add nodes
workflow.add_node("research_node", research_node)
workflow.add_node("validate_node", validate_node)
workflow.add_node("report_node", report_node)
workflow.add_node("reject_node", reject_node)

# Add edges
workflow.add_edge(START, "research_node")
workflow.add_edge("research_node", "validate_node")
workflow.add_conditional_edges(
    "validate_node",
    route_after_validation,
    {
        "report_node": "report_node",
        "reject_node": "reject_node"
    }
)
workflow.add_edge("report_node", END)
workflow.add_edge("reject_node", END)

app = workflow.compile()
print("✅ Graph compiled")

In [ ]:
# ── 5. Run It ─────────────────────────────────────────────────────────────────

console.print(Panel(
    "[bold]Knovik Niche Research Agent[/]\nPowered by LangGraph + Claude",
    style="cyan"
))

result = app.invoke({
    "niche": "Zigbee smart home hubs for Apple HomeKit users"
})

# Display summary table
table = Table(title="Research Summary", show_header=True)
table.add_column("Field", style="cyan")
table.add_column("Value")
table.add_row("Niche", result["niche"])
table.add_row("Viability Score", f"{result['viability_score']}/100")
table.add_row("Decision", f"[green]{result['decision'].upper()}[/]" if result['decision'] == 'pursue' else f"[red]{result['decision'].upper()}[/]")
table.add_row("Reasoning", result["score_reasoning"])
console.print(table)

if result.get("final_report") and not result["final_report"].startswith("REJECTED"):
    console.print(Panel(result["final_report"], title="📄 Full Report", border_style="green"))

## Exercises

1. **Change the niche**: Try `"Matter protocol bridge devices"` or `"HomeKit-compatible security cameras"`
2. **Add a node**: Insert a `competitive_analysis_node` between `research_node` and `validate_node`
3. **Lower the threshold**: Change the rejection condition to `score < 30` and see what niches now get through
4. **Add human-in-the-loop**: Before `report_node`, add a checkpoint that asks for human approval — see {doc}`04_human_in_the_loop`

---

## What LangGraph gave us

- **Typed state** shared across all nodes — no passing arguments manually
- **Conditional routing** based on runtime values — not possible in a linear chain
- **Clear graph structure** you can visualize and reason about
- **Built-in checkpointing** for persistence (not used here, see chapter 4)

Next: See the same task in {doc}`../crewai/05_demo_content_crew` to understand when CrewAI's approach is more natural.